## Co policzyć

- Czy istnieje zależność płci od cech (featerów)? Jeśli tak, to jakich?  
  → Jak znaleźć przelicznik/miarę tej zależności?

- Ogólna wielkość muszli – jak zależy od parametrów takich jak szerokość, wysokość itd.?  
  → Czy te cechy zachowują się regularnie?

- Jaka jest zależność między masą mięsa a wielkością muszli?

- Jak ma się całkowita masa do masy mięsa?

---

## Co trzeba sprawdzić

- Jak zmienia się masa mięsa w muszli?  
  → Czy istnieje limit jej wzrostu?

- Jak dokładnie mierzona jest masa mięsa (`weight of meat`)?

- Czym jest `gut weight` w przypadku muszli?  
  → Może chodzi o mięsień utrzymujący muszlę? Jelita? (czy jest coś jak "krwawienie jelit"?)

- Błąd pomiarowy – jak go oszacować albo sprawdzić?

- Jakie są używane metody pomiarowe?

---

## TODO dla mnie

- Sprawdzić liniowość między zmiennymi

- Zbadać zależność cech geometrycznych

- Przeanalizować rozkład mięsa i masy (czy istnieje między nimi jakaś zależność)


In [ ]:
# basic 
import pandas as pd
import numpy as np
import scipy.stats as stasts

# visualization
import matplotlib as plt
import seaborn as sns

# statistics
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
from statsmodels.tools.tools import add_constant

# models 
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
# from sklearn.metrics import 

# data
from ucimlrepo import fetch_ucirepo 

# global parameters
seed = 1333

load data

In [ ]:
# fetch dataset 
abalone = fetch_ucirepo(id=1) 

# data (as pandas dataframes) 
X = abalone.data.features 
y = abalone.data.targets 
df = pd.concat([X, y], axis=1)

# saving data 
# df.to_csv('abalone.csv', index=False)
# df.to_excel('abalone.xlsx')

In [ ]:
# parameters 
col_names = ['Płeć', 'Długość', 'Średnica', 'Wysokość', 'Cała_waga', 'Waga_po_obraniu', 'Waga_trzewi', 'Waga_powłoki', 'Pierścienie']

# loading data
df = pd.read_csv('abalone.csv', names=col_names, header=0)
X = df.drop('Pierścienie', axis=1)
y = df[['Pierścienie']]

# decoding categorical data 
X['Płeć_M'] = (X['Płeć'] == 'M').astype(int)
X['Płeć_F'] = (X['Płeć'] == 'F').astype(int)
X['Płeć_I'] = (X['Płeć'] == 'I').astype(int)
X = X[list(X.columns[-3:]) + list(X.columns[1:-3])]

# X = pd.get_dummies(X, columns=['Płeć'], drop_first=True)


analiza korelacji 

In [ ]:
# ------- creating linear model ------- #
model_with_sex = make_pipeline(
    LinearRegression()
)
model_with_sex.fit(X, y)

model_without_sex = make_pipeline(
    LinearRegression()
)
# model_without_sex.fit(X.drop(columns=['Płeć_M', 'Płeć_F']), y)

# ------- correlation ------- #
sns.heatmap(X.drop(columns=['Płeć_I', 'Płeć_M', 'Płeć_F']).corr(), annot=True, cmap='coolwarm')

# ------- vif ------- #
X_vif = add_constant(X)
# X.columns
pd.Series([vif(X_vif, i) for i in range(1+3, X_vif.shape[1])], index=X.columns[3:])

# print(vif(X.drop(columns=['Płeć_M', 'Płeć_F'])))


próbne modele regresji 

In [ ]:

# splitting data
# X.drop(['Sex'], axis=1)
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X_encoded, y, test_size=0.2, random_state=seed)
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_encoded, y, test_size=0.2, random_state=seed)

# calculating model with sex
model_1 = make_pipeline(
    StandardScaler(),
    LinearRegression()
    )
model_1.fit(X_train_1, y_train_1)
model_1.score(X_test_1, y_test_1)
# calculating model without sex
model_2 = make_pipeline(
    StandardScaler(),
    LinearRegression()
)
model_2.fit()

# X_encoded

# ważne tej shave jest bardzo ważny 